# PyTorch CIFAR10 Classifier

![PyTorch Logo](https://saturn-public-assets.s3.us-east-2.amazonaws.com/example-resources/pytorch-logo.png)

This example demonstrates GPU-accelerated image classification using PyTorch on Saturn Cloud, running on GPU Jupyter server resource. 

The code trains a ResNet18 neural network on the CIFAR10 dataset to classify images into 10 categories: airplane, car, bird, cat, deer, dog, frog, horse, ship, and truck. ResNet18 uses "residual connections" that enable the network to learn complex visual patterns while avoiding the vanishing gradient problem.

Leverage the use of multiple GPU on [Saturn Cloud](https://saturncloud.io/?utm_source=Blog+&utm_medium=Try&utm_campaign=Try) to effectively decrease model’s training time and handle larger datasets. [Check our blog](https://saturncloud.io/blog/how-to-use-multiple-gpus-in-pytorch/) on exploring the computational power of multiple GPUs in PyTorch.

## Installing Dependencies
First, ensure all required packages are installed.

In [ ]:
!pip install -q -r requirements.txt

## Import Libraries and Setup
This code mainly relies on PyTorch and Torchvision for the deep learning work.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import numpy as np
import pandas as pd
import hvplot.pandas
import panel as pn
import matplotlib.pyplot as plt

# Enable Panel extension for dashboard
pn.extension()

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Preparing Data

This code is used to get the CIFAR10 dataset and format it properly for training, also define the image transformations. 

In [ ]:
# Training transforms with data augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Test transforms without augmentation
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

Next, download the CIFAR10 dataset and create data loaders. The dataset will be automatically downloaded on first run (~170MB).

In [ ]:
# Download and load training data
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2
)

# Download and load test data
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2
)

# Class names for visualization
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training samples: {len(trainset)}")
print(f"Test samples: {len(testset)}")

### Define the Model Architecture

This code defines the ResNet18 structure that the neural network will use. ResNet18 is a convolutional neural network with 18 layers that uses residual connections to enable training of deeper networks.

In [ ]:
# Load ResNet18 and modify for CIFAR10 (10 classes)
model = resnet18(weights=None, num_classes=10)
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model initialized and ready for training")

### Train the Model

We define a `train()` function that will do the work to train the neural network. This function trains the model for a specified number of epochs and tracks both training and test accuracy. It uses the GPU via `device` for acceleration.

In [ ]:
def train(model, trainloader, testloader, criterion, optimizer, device, num_epochs=5):
    train_losses = []
    train_accuracies = []
    test_accuracies = []
    
    print("Starting training...\n")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        # Training loop
        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        # Calculate training metrics
        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)
        
        # Evaluate on test set
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100. * test_correct / test_total
        test_accuracies.append(test_acc)
        
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Loss: {epoch_loss:.3f} | '
              f'Train Acc: {epoch_acc:.2f}% | '
              f'Test Acc: {test_acc:.2f}%')
    
    print("\nTraining completed!")
    return train_losses, train_accuracies, test_accuracies

The next block of code actually runs the training function and creates the trained model.

In [ ]:
num_epochs = 5
train_losses, train_accuracies, test_accuracies = train(
    model, trainloader, testloader, criterion, optimizer, device, num_epochs
)

## Visualizing Results
### Generate Training Curves
To visualize the training progress, this create interactive plots showing loss and accuracy over time.

In [ ]:
# Create dataframe with training metrics
curves_data = pd.DataFrame({
    'Epoch': range(1, num_epochs + 1),
    'Training Loss': train_losses,
    'Training Accuracy': train_accuracies,
    'Test Accuracy': test_accuracies
})

# Create loss curve
loss_curve = curves_data.hvplot.line(
    x='Epoch', 
    y='Training Loss', 
    title='Training Loss Over Time',
    color='#e74c3c',
    line_width=3,
    width=600,
    height=300
)

# Create accuracy curves
acc_curve = curves_data.hvplot.line(
    x='Epoch', 
    y=['Training Accuracy', 'Test Accuracy'],
    title='Accuracy Over Time',
    line_width=3,
    width=600,
    height=300,
    legend='top_left'
)

# Display curves
pn.Column(loss_curve, acc_curve)

## Create Interactive Dashboard

Finally, we can create an interactive dashboard that displays all results in one place. This dashboard can be deployed to Saturn Cloud for continuous hosting.

In [ ]:
# Helper function to create KPI boxes
def kpi_box(title, color, value, unit=""):
    return pn.pane.Markdown(
        f"""
        ### {title}
        # {value}{unit}
        """,
        styles={
            "background-color": "#F6F6F6",
            "border": f"2px solid {color}",
            "border-radius": "5px",
            "padding": "10px",
            "color": color,
        },
    )

# Final metric values
final_test_acc = test_accuracies[-1]
final_train_acc = train_accuracies[-1]

# Create KPI boxes
test_acc_kpi = kpi_box("Final Test Accuracy", "#27ae60", f"{final_test_acc:.2f}", "%")
train_acc_kpi = kpi_box("Final Train Accuracy", "#3498db", f"{final_train_acc:.2f}", "%")
epochs_kpi = kpi_box("Epochs Trained", "#9b59b6", num_epochs)

# Dashboard introduction content
dashboard_intro = """
# CIFAR10 Image Classification Dashboard

This dashboard shows the results of training a **ResNet18** model on the CIFAR10 dataset using GPU acceleration.
The model was trained to classify images into 10 categories: plane, car, bird, cat, deer, dog, frog, horse, ship, and truck.

## About the Model

- **Architecture:** ResNet18 (18-layer residual network)
- **Dataset:** CIFAR10 (60,000 images, 10 classes)
- **Training:** GPU-accelerated with PyTorch
"""

# Deployment information
about_deployment = """
## Deploying on Saturn Cloud

This dashboard can be deployed to Saturn Cloud for continuous hosting, allowing users without notebook access 
to view the results. The model training leverages **NVIDIA GPU acceleration** for faster training times.  
Learn more about GPU deployments in the [Saturn Cloud documentation](https://saturncloud.io/docs/).
"""

# Convert Matplotlib figure to Panel pane
image_grid = pn.pane.Matplotlib(fig, tight=True)

# Create dashboard layout using GridSpec
dashboard = pn.GridSpec(
    name="CIFAR10 Dashboard",
    sizing_mode="stretch_both",
    min_width=900,
    min_height=800
)

# Layout configuration
dashboard[0:3, 0:2] = pn.Column(dashboard_intro, about_deployment)
dashboard[0, 2] = test_acc_kpi
dashboard[0, 3] = train_acc_kpi
dashboard[0, 4] = epochs_kpi
dashboard[1:3, 2:5] = loss_curve
dashboard[3:5, 0:5] = acc_curve
dashboard[5:9, 0:5] = image_grid


In [ ]:
# Display the dashboard
dashboard

If you wanted to experiment with trying many different hyperparameters for the model, you could concurrently train models with different hyperparameters using [distributed computing](https://github.com/saturncloud/examples/blob/54589b72f63737afd2ccce5a38b7feac86b17c96/examples/pytorch/02-pytorch-gpu-dask-multiple-models.ipynb). You could also train a single neural network over many GPUs at once with distributed computing via PyTorch's DistributedDataParallel or Dask.